# Prefill–Decode 解耦：用请求 Trace 做路由与降级

**面试问题：为什么拆分 prefill/decode，KV 传输、节点选择、SLO 和故障降级怎样权衡？**

## 回答主线

先把真实请求和资源合同摆出来，用最简单的方案建立成本或正确性基线，再手写核心控制逻辑并展示完整事件、指标和失败修正。断言只在最后保护少量关键不变量，前面的可见输入、过程和结果才是学习主体。

## 真实案例

平台同时收到长文档摘要、短聊天和代码审查请求。Prefill 偏计算，decode 偏显存带宽；两类节点负载不同，但拆分会增加 KV 网络传输。案例用六条真实字段 trace 对比共置基线和 P/D 路由，展示 TTFT、TPOT、goodput、节点队列与一个慢网络下必须回退共置的反例。

### 输入预览：六条混合长度请求

In [1]:
requests = [  # 构造同时包含长 prompt、短聊天和不同输出长度的请求 trace。
    {"id": "doc-1", "kind": "文档摘要", "prompt": 6000, "decode": 180, "ttft_slo": 1.8, "tpot_slo": 0.055},  # 长文档适合计算型 prefill 节点。
    {"id": "chat-1", "kind": "在线客服", "prompt": 220, "decode": 90, "ttft_slo": 0.45, "tpot_slo": 0.045},  # 短聊天可能不值得支付 KV 传输开销。
    {"id": "code-1", "kind": "代码审查", "prompt": 3400, "decode": 320, "ttft_slo": 1.4, "tpot_slo": 0.060},  # 中长 prompt 和长 decode 需要两类资源。
    {"id": "chat-2", "kind": "在线客服", "prompt": 180, "decode": 60, "ttft_slo": 0.40, "tpot_slo": 0.045},  # 第二条短请求用于观察共置回退。
    {"id": "doc-2", "kind": "合同问答", "prompt": 7200, "decode": 120, "ttft_slo": 2.0, "tpot_slo": 0.055},  # 超长 prompt 对 prefill 吞吐敏感。
    {"id": "agent-1", "kind": "Agent规划", "prompt": 1500, "decode": 420, "ttft_slo": 0.9, "tpot_slo": 0.065},  # 长 decode 对 decode 节点队列敏感。
]  # 完成请求输入列表。
print("请求ID   类型       prompt decode  TTFT_SLO TPOT_SLO")  # 输出 trace 表头。
for request in requests:  # 逐条展示路由器可以观察的请求字段。
    print(f'{request["id"]:<8} {request["kind"]:<8} {request["prompt"]:>6} {request["decode"]:>6} {request["ttft_slo"]:>8.2f} {request["tpot_slo"]:>8.3f}')  # 显示长度和两个不同 SLO。

请求ID   类型       prompt decode  TTFT_SLO TPOT_SLO
doc-1    文档摘要       6000    180     1.80    0.055
chat-1   在线客服        220     90     0.45    0.045
code-1   代码审查       3400    320     1.40    0.060
chat-2   在线客服        180     60     0.40    0.045
doc-2    合同问答       7200    120     2.00    0.055
agent-1  Agent规划    1500    420     0.90    0.065


## Baseline 基线：所有阶段共置在通用节点

In [2]:
def colocated_estimate(request, queue_seconds=0.18):  # 实现共置节点的可解释延迟模型。
    prefill_seconds = request["prompt"] / 5200.0  # 用通用节点每秒五千二百 prefill token 估算计算时间。
    tpot = 0.043 + 0.000002 * request["prompt"]  # 用 prompt 占用 KV 对 decode 带宽的影响估算 TPOT。
    ttft = queue_seconds + prefill_seconds  # 把排队和 prefill 计算合成首 token 延迟。
    return {"route": "colocated", "ttft": ttft, "tpot": tpot, "transfer": 0.0}  # 返回基线延迟分解。

baseline = {request["id"]: colocated_estimate(request) for request in requests}  # 对六条请求计算共置基线。
print("共置基线延迟：")  # 输出基线结果标题。
for request in requests:  # 逐请求展示 TTFT 和 TPOT。
    result = baseline[request["id"]]  # 读取当前请求的基线估计。
    print(f'{request["id"]:<8} TTFT={result["ttft"]:.3f}s TPOT={result["tpot"]:.3f}s')  # 展示长 prompt 同时伤害首 token 和 decode。

共置基线延迟：
doc-1    TTFT=1.334s TPOT=0.055s
chat-1   TTFT=0.222s TPOT=0.043s
code-1   TTFT=0.834s TPOT=0.050s
chat-2   TTFT=0.215s TPOT=0.043s
doc-2    TTFT=1.565s TPOT=0.057s
agent-1  TTFT=0.468s TPOT=0.046s


### 核心实现：阶段专用节点、KV 传输与准入

In [3]:
prefill_nodes = [{"id": "p0", "queue": 0.05, "rate": 9500.0}, {"id": "p1", "queue": 0.12, "rate": 11000.0}]  # 定义两台计算优化的 prefill 节点。
decode_nodes = [{"id": "d0", "queue": 0.08, "base_tpot": 0.031}, {"id": "d1", "queue": 0.03, "base_tpot": 0.035}]  # 定义两台带宽优化的 decode 节点。

def transfer_seconds(request, bandwidth_gbps=80.0, layers=32, kv_heads=8, head_dim=128):  # 估算 prefill 生成的 KV 跨节点传输时间。
    kv_bytes = 2 * layers * request["prompt"] * kv_heads * head_dim * 2  # 统计 K/V、层、token、head、维度和 FP16 字节。
    return kv_bytes / (bandwidth_gbps * 1e9 / 8.0)  # 用网络有效字节带宽转换成秒。

def disaggregated_route(request, bandwidth_gbps=80.0):  # 为单请求选择 P/D 节点或共置回退。
    if request["prompt"] < 512:  # 短 prompt 的 KV 传输和跨节点调度收益通常不足。
        result = colocated_estimate(request, queue_seconds=0.08)  # 使用低队列共置池处理短聊天。
        result["route"] = "fallback-colocated"  # 明确记录这是有意降级而非路由失败。
        return result  # 返回短请求共置结果。
    prefill = min(prefill_nodes, key=lambda node: node["queue"] + request["prompt"] / node["rate"])  # 选择预计完成最早的 prefill 节点。
    decode = min(decode_nodes, key=lambda node: node["queue"] + node["base_tpot"] * request["decode"])  # 选择当前 decode 完成时间最短的节点。
    transfer = transfer_seconds(request, bandwidth_gbps=bandwidth_gbps)  # 计算当前网络条件下的 KV 搬运成本。
    ttft = prefill["queue"] + request["prompt"] / prefill["rate"] + transfer + decode["queue"]  # 汇总 prefill 排队、计算、传输和 decode 准备时间。
    tpot = decode["base_tpot"]  # 使用带宽优化节点的稳定 token 间延迟。
    return {"route": f'{prefill["id"]}->{decode["id"]}', "ttft": ttft, "tpot": tpot, "transfer": transfer}  # 返回完整路由和延迟分解。

routed = {request["id"]: disaggregated_route(request) for request in requests}  # 对六条请求执行 P/D 路由。
print("P/D 路由决策：")  # 输出路由结果标题。
for request in requests:  # 逐请求展示节点和传输成本。
    result = routed[request["id"]]  # 读取当前请求的路由结果。
    print(f'{request["id"]:<8} route={result["route"]:<18} transfer={result["transfer"]:.3f}s TTFT={result["ttft"]:.3f}s TPOT={result["tpot"]:.3f}s')  # 展示阶段拆分的收益和代价。

P/D 路由决策：
doc-1    route=p1->d0             transfer=0.079s TTFT=0.824s TPOT=0.031s
chat-1   route=fallback-colocated transfer=0.000s TTFT=0.122s TPOT=0.043s
code-1   route=p0->d0             transfer=0.045s TTFT=0.532s TPOT=0.031s
chat-2   route=fallback-colocated transfer=0.000s TTFT=0.115s TPOT=0.043s
doc-2    route=p1->d0             transfer=0.094s TTFT=0.949s TPOT=0.031s
agent-1  route=p0->d0             transfer=0.020s TTFT=0.308s TPOT=0.031s


## 结果解读：逐请求 SLO 与 Goodput

In [4]:
def meets_slo(request, result):  # 判断请求的首 token 和 token 间延迟是否同时达标。
    return result["ttft"] <= request["ttft_slo"] and result["tpot"] <= request["tpot_slo"]  # 只有两个 SLO 都满足才计入 goodput。

baseline_good = sum(meets_slo(request, baseline[request["id"]]) for request in requests)  # 统计共置基线达标请求数。
routed_good = sum(meets_slo(request, routed[request["id"]]) for request in requests)  # 统计 P/D 路由达标请求数。
print("请求     共置达标  P/D达标  TTFT变化  TPOT变化")  # 输出逐请求 SLO 对比表头。
for request in requests:  # 逐条解释阶段拆分对两类延迟的影响。
    before = baseline[request["id"]]  # 读取共置基线。
    after = routed[request["id"]]  # 读取路由结果。
    print(f'{request["id"]:<8} {str(meets_slo(request, before)):<8} {str(meets_slo(request, after)):<7} {after["ttft"] - before["ttft"]:+8.3f}s {after["tpot"] - before["tpot"]:+8.3f}s')  # 显示 TTFT 与 TPOT 可能方向不同。
print(f"Goodput 达标请求：共置 {baseline_good}/{len(requests)} -> P/D {routed_good}/{len(requests)}")  # 用 SLO 达标数而非裸吞吐评价方案。

请求     共置达标  P/D达标  TTFT变化  TPOT变化
doc-1    True     True      -0.510s   -0.024s
chat-1   True     True      -0.100s   +0.000s
code-1   True     True      -0.301s   -0.019s
chat-2   True     True      -0.100s   +0.000s
doc-2    False    True      -0.616s   -0.026s
agent-1  True     True      -0.161s   -0.015s
Goodput 达标请求：共置 5/6 -> P/D 6/6


## 失败案例：网络拥塞时长 Prompt 也不该强制拆分

In [5]:
slow_network = {request["id"]: disaggregated_route(request, bandwidth_gbps=4.0) for request in requests}  # 把 KV 网络带宽从八十降到四 Gbps 模拟拥塞。
adaptive = {}  # 保存比较慢网 P/D 与共置后选择的安全路径。
for request in requests:  # 逐请求执行收益感知的动态降级。
    candidate = slow_network[request["id"]]  # 读取慢网下的拆分候选。
    colocated = baseline[request["id"]]  # 读取稳定共置路径。
    adaptive[request["id"]] = candidate if candidate["ttft"] < colocated["ttft"] else dict(colocated, route="network-fallback")  # 只有 TTFT 真正更低时才接受拆分。
print("慢网络下的失败与修正：")  # 输出降级结果标题。
for request in requests:  # 展示长 prompt 的传输成本和最终选择。
    slow = slow_network[request["id"]]  # 读取慢网候选用于对比。
    fixed = adaptive[request["id"]]  # 读取自适应降级结果。
    print(f'{request["id"]:<8} slow-transfer={slow["transfer"]:.3f}s candidate-TTFT={slow["ttft"]:.3f}s final={fixed["route"]}')  # 展示 P/D 不是无条件更快。

慢网络下的失败与修正：
doc-1    slow-transfer=1.573s candidate-TTFT=2.318s final=network-fallback
chat-1   slow-transfer=0.000s candidate-TTFT=0.122s final=fallback-colocated
code-1   slow-transfer=0.891s candidate-TTFT=1.379s final=network-fallback
chat-2   slow-transfer=0.000s candidate-TTFT=0.115s final=fallback-colocated
doc-2    slow-transfer=1.887s candidate-TTFT=2.742s final=network-fallback
agent-1  slow-transfer=0.393s candidate-TTFT=0.681s final=network-fallback


### 生产边界

In [6]:
router_snapshot = {"prefill_nodes": [node["id"] for node in prefill_nodes], "decode_nodes": [node["id"] for node in decode_nodes], "bandwidth_gbps": 80.0, "goodput": f"{routed_good}/{len(requests)}", "fallback_policy": "compare-expected-ttft"}  # 构造可发布和回放的路由策略快照。
print("路由快照：", router_snapshot)  # 展示线上决策必须绑定的容量和策略版本。
print("生产替换点：还需真实排队预测、KV 序列化、RDMA、背压、节点故障、prefix cache locality 和分租户公平。")  # 明确教学延迟模型与真实集群控制面的差距。

路由快照： {'prefill_nodes': ['p0', 'p1'], 'decode_nodes': ['d0', 'd1'], 'bandwidth_gbps': 80.0, 'goodput': '6/6', 'fallback_policy': 'compare-expected-ttft'}
生产替换点：还需真实排队预测、KV 序列化、RDMA、背压、节点故障、prefix cache locality 和分租户公平。


## 回归测试：只保护路由、SLO 和降级

In [7]:
assert routed["chat-1"]["route"] == "fallback-colocated"  # 验证短 prompt 不会无收益地跨节点搬运 KV。
assert routed["doc-1"]["route"].startswith("p")  # 验证长文档进入专用 prefill/decode 路径。
assert routed_good >= baseline_good  # 验证目标网络条件下 P/D 路由没有降低 SLO goodput。
assert any(result["route"] == "network-fallback" for result in adaptive.values())  # 验证慢网络能触发共置降级。
assert all(result["ttft"] <= slow_network[key]["ttft"] + 1e-12 for key, result in adaptive.items())  # 验证自适应策略不会比慢网候选更差。
print("回归测试通过：短请求回退、长请求拆分、goodput 和网络降级均符合合同。")  # 用少量断言总结路由正确性。

回归测试通过：短请求回退、长请求拆分、goodput 和网络降级均符合合同。
